# 01 — Download and validate SciFact
This notebook downloads the official SciFact release from the URL published in the AllenAI repository. It records the archive checksum and validates claim/corpus schemas. The test split is unlabeled; the labelled dev split is used for evaluation.

In [ ]:
from pathlib import Path
import urllib.request, tarfile, hashlib, json, os
BASE = Path("/content/scifact_study")
DATA = BASE / "data"
DATA.mkdir(parents=True, exist_ok=True)
URL = "https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz"
archive = DATA / "scifact_data.tar.gz"
if not archive.exists():
    print("Downloading SciFact release...")
    urllib.request.urlretrieve(URL, archive)
sha256 = hashlib.sha256(archive.read_bytes()).hexdigest()
print("Archive:", archive, "bytes:", archive.stat().st_size, "SHA256:", sha256)


In [ ]:
# Extract once
extract_dir = DATA / "release"
extract_dir.mkdir(exist_ok=True)
if not any(extract_dir.rglob("corpus.jsonl")):
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(extract_dir, filter="data")
corpus_paths = list(extract_dir.rglob("corpus.jsonl"))
assert corpus_paths, "Could not locate corpus.jsonl after extraction"
CORPUS_PATH = corpus_paths[0]
claims_dev_paths = list(extract_dir.rglob("claims_dev.jsonl"))
claims_train_paths = list(extract_dir.rglob("claims_train.jsonl"))
claims_test_paths = list(extract_dir.rglob("claims_test.jsonl"))
assert claims_dev_paths and claims_train_paths and claims_test_paths
DEV_PATH, TRAIN_PATH, TEST_PATH = claims_dev_paths[0], claims_train_paths[0], claims_test_paths[0]
print("Corpus:", CORPUS_PATH)
print("Train:", TRAIN_PATH)
print("Dev:", DEV_PATH)
print("Test:", TEST_PATH)


In [ ]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]
corpus = read_jsonl(CORPUS_PATH)
train = read_jsonl(TRAIN_PATH)
dev = read_jsonl(DEV_PATH)
test = read_jsonl(TEST_PATH)
print("Corpus abstracts:", len(corpus))
print("Train claims:", len(train), "Dev claims:", len(dev), "Test claims:", len(test))
print("Corpus keys:", sorted(corpus[0].keys()))
print("Claim keys:", sorted(dev[0].keys()))
print("First dev claim:", dev[0]["id"], dev[0]["claim"])


In [ ]:
# Schema and ID checks
doc_ids = [int(d["doc_id"]) for d in corpus]
assert len(doc_ids) == len(set(doc_ids)), "Duplicate corpus doc_id values"
for split_name, rows in [("train", train), ("dev", dev), ("test", test)]:
    ids = [int(r["id"]) for r in rows]
    assert len(ids) == len(set(ids)), f"Duplicate claim IDs in {split_name}"
    for r in rows:
        assert "claim" in r and isinstance(r["claim"], str)
        if split_name != "test":
            assert "evidence" in r and isinstance(r["evidence"], dict)
            for doc_id, evs in r["evidence"].items():
                assert int(doc_id) in set(doc_ids)
                for ev in evs:
                    assert ev["label"] in {"SUPPORT", "CONTRADICT"}
                    assert all(isinstance(i, int) for i in ev["sentences"])
print("Schema checks passed.")
print("Test split has evidence labels:", "evidence" in test[0])


In [ ]:
# Save paths and dataset provenance for later notebooks
import json, hashlib
manifest = {
    "corpus_path": str(CORPUS_PATH), "train_path": str(TRAIN_PATH),
    "dev_path": str(DEV_PATH), "test_path": str(TEST_PATH),
    "archive_sha256": sha256,
    "counts": {"corpus": len(corpus), "train": len(train), "dev": len(dev), "test": len(test)}
}
(BASE / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))


## Important
Use the dev split for labelled comparisons. Do not calculate labelled test metrics: the official test labels are not released.